# Hy-MT3 1.8B — multilingual translation on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Hy-MT3-1.8B-translation.ipynb)

[`tencent/Hy-MT3-1.8B`](https://huggingface.co/tencent/Hy-MT3-1.8B) is Tencent's **1.8B** checkpoint in the third-generation Hy translation family (Apache-2.0, public). The card lists three sizes — 1.8B, 7B, and 30B-A3B (MoE) — and **102 languages**. Architecture is `hunyuan_v1_dense` (`HunYuanDenseV1ForCausalLM`). Hub weights are one **3.6 GB** `model.safetensors` shard. FP8 and GGUF siblings exist; this notebook loads the **full 1.8B safetensors in float16**.

**Why this model.** A small Apache-2.0 text translator with the card's constrained-translation prompts (terminology, style, delimiters, structured data), not a general chat model. The card: no default system prompt; use **full language names** in the prompt.

See [Use cases](https://github.com/donvito/notebooks/blob/main/docs/Hy-MT3-1.8B-translation.md) for when this 1.8B is a good fit and when it is not.

**Licence:** Apache-2.0. Weights are public — `HF_TOKEN` is optional.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about 15 GB of VRAM, **no native bfloat16**, and **no FlashAttention-2**. The card's Transformers snippet loads `dtype=torch.bfloat16`; this notebook loads **float16**. The card pins `transformers>=5.6.0`. **Do not reinstall torch.** Card `max_context` is 8192; demos stay short.

**Not in this notebook:** 7B / 30B-A3B, FP8, GGUF, vLLM / SGLang serving, or a rerun of TransRubric / WildMTBench numbers.

**Links:** [model card](https://huggingface.co/tencent/Hy-MT3-1.8B) · [use cases](https://github.com/donvito/notebooks/blob/main/docs/Hy-MT3-1.8B-translation.md) · [collection](https://huggingface.co/collections/tencent/hy-mt3) · [GitHub](https://github.com/Tencent-Hunyuan/Hy-MT3) · [report](https://arxiv.org/abs/2611.27673) · [announcement](https://x.com/TencentAI_News/status/2125177755943960580) · [GGUF](https://huggingface.co/tencent/Hy-MT3-1.8B-GGUF) · [FP8](https://huggingface.co/tencent/Hy-MT3-1.8B-FP8)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so a later transformers pin can take effect without a restart. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention-2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Checkpoint dtype is bfloat16. T4 is Turing sm_75 — stay on float16.
DTYPE = torch.float16
free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16 (~3.6 GB weights).")


## 2. Install Transformers

The [card](https://huggingface.co/tencent/Hy-MT3-1.8B) requires `transformers>=5.6.0` and loads `AutoTokenizer` + `AutoModelForCausalLM` with `trust_remote_code=True`. Also install `accelerate` and **`jinja2>=3.1.6`** (`apply_chat_template` needs it). Use **`%pip`** (not `python -I -m pip`). Isolated mode drops Colab's site-packages and breaks the preinstalled `torch`. Keep Colab's torch. Do **not** install `flash-attn`. The install cell then drops any already-imported `transformers` modules so **Runtime > Run all** can pick up the pin without *Restart session*. If the version assert still fails, restart once and continue from the GPU cell.


In [ ]:
%pip install -q -U "transformers>=5.6.0" "accelerate>=1.1.0" "jinja2>=3.1.6"

import sys

# Card: transformers>=5.6.0, AutoTokenizer + AutoModelForCausalLM, trust_remote_code.
# jinja2>=3.1.6 is required for apply_chat_template.
# Do NOT install flash-attn (T4 cannot run FlashAttention-2).
# Do NOT replace Colab's CUDA torch wheel.

for name in list(sys.modules):
    if name == "transformers" or name.startswith("transformers."):
        del sys.modules[name]

import transformers
from packaging.version import Version

assert Version(transformers.__version__.split("+")[0]) >= Version("5.6.0"), (
    f"transformers {transformers.__version__} is loaded; Hy-MT3-1.8B needs "
    "transformers>=5.6.0. Runtime > Restart session, then rerun from the GPU check cell."
)
print("transformers", transformers.__version__)


## 3. Load Hy-MT3-1.8B

`AutoTokenizer` + `AutoModelForCausalLM`, as in the [card's Transformers snippet](https://huggingface.co/tencent/Hy-MT3-1.8B). `trust_remote_code=True`. First run downloads about **3.6 GB**. Attention is **sdpa**, then **eager**. No 4-bit: float16 1.8B fits a 15 GB T4.

The helper fills the card's **English** prompt templates (full language names). Sampling is the card's 1.8B recommendation: `temperature=0.7`, `top_p=0.6`, `top_k=20`, `repetition_penalty=1.05`. The card's `max_tokens` is 4096; demos use a few hundred new tokens. No system prompt.


In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "tencent/Hy-MT3-1.8B"

# Card, 1.8B / 7B inference parameters.
SAMPLER = dict(
    do_sample=True,
    temperature=0.7,
    top_p=0.6,
    top_k=20,
    repetition_penalty=1.05,
)

DEFAULT_PROMPT = (
    "Translate the following text into {target_lang}. "
    "Note that you should **only output the translated result without any additional explanation**:\n"
    "\n"
    "{source_text}"
)

TERMINOLOGY_PROMPT = (
    "Reference the following translations:\n"
    "{glossary}\n"
    "\n"
    "Translate the following text into {target_lang}. "
    "Note that you must **ONLY output the translated result without any additional explanation**:\n"
    "\n"
    "{source_text}"
)


def print_vram(label=""):
    suffix = f" ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM{suffix}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"peak {peak:.2f} GB | free {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


def load_causal(attn):
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        attn_implementation=attn,
        device_map="auto",
        trust_remote_code=True,
    ).eval()


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

ATTN = None
model = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_causal(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} float16 load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("could not load tencent/Hy-MT3-1.8B") from last_error

n_params = sum(p.numel() for p in model.parameters()) / 1e9
print("attention:", ATTN, "| dtype:", next(model.parameters()).dtype)
print(f"parameters: {n_params:.2f}B")
print_vram("after load")


def default_prompt(source_text, target_lang):
    return DEFAULT_PROMPT.format(target_lang=target_lang, source_text=source_text)


def terminology_prompt(source_text, target_lang, pairs):
    glossary = "\n".join(f"{src} translates to {tgt}" for src, tgt in pairs)
    return TERMINOLOGY_PROMPT.format(
        glossary=glossary,
        target_lang=target_lang,
        source_text=source_text,
    )


def translate(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            **SAMPLER,
        )
    text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    )
    print_vram("after generate")
    return text.strip()


## 4. Demo 1 — English → Chinese, Spanish, Japanese

Card **Default Translation** template, English prompt, full language names. Three short sentences, each into Chinese, Spanish, and Japanese. Output is translation only.


In [ ]:
SENTENCES = [
    "The weather is nice today.",
    "Please wait while we process your request.",
    "The library opens at nine in the morning.",
]
TARGETS = ["Chinese", "Spanish", "Japanese"]

for sentence in SENTENCES:
    print("=" * 72)
    print("EN:", sentence)
    for lang in TARGETS:
        prompt = default_prompt(sentence, lang)
        print(f"\n--- → {lang} ---")
        print(translate(prompt, max_new_tokens=128))
    print()


## 5. Demo 2 — less common pairs (Filipino, Swahili)

Same default template. The card lists **Filipino** (`tl`, supported since Hy-MT2) and **Swahili** (`sw`, new in Hy-MT3) among the 102 languages. Full English names, as the card requires for English prompts.


In [ ]:
LOW_RESOURCE = [
    "The farmers planted rice before the rainy season.",
    "Please show your ticket at the station entrance.",
]
LOW_TARGETS = ["Filipino", "Swahili"]

for sentence in LOW_RESOURCE:
    print("=" * 72)
    print("EN:", sentence)
    for lang in LOW_TARGETS:
        prompt = default_prompt(sentence, lang)
        print(f"\n--- → {lang} ---")
        print(translate(prompt, max_new_tokens=128))
    print()


## 6. Demo 3 — terminology-constrained translation

Card **Terminology** template: glossary lines of the form `{text} translates to {text}`, then the default “ONLY output the translated result” instruction. One short UI / game string into Chinese. This is the card's format, not a rerun of the Honor of Kings evaluation.


In [ ]:
PAIRS = [
    ("Honor of Kings", "王者荣耀"),
    ("Start button", "开始按钮"),
    ("tutorial match", "人机对战"),
]
SOURCE = "Tap the Start button to begin the Honor of Kings tutorial match."

prompt = terminology_prompt(SOURCE, "Chinese", PAIRS)
print("prompt:")
print(prompt)
print()
print("--- translation ---")
print(translate(prompt, max_new_tokens=128))


## 7. Caveats and cleanup

- **Prompt templates.** Default and terminology strings match the English column of the card's translation-task table. Use full language names. Chinese prompts would use Chinese language names; this notebook stays on English prompts. No system prompt (the card: none by default).
- **Sampler.** 1.8B recommendation on the card: `temperature=0.7`, `top_p=0.6`, `top_k=20`, `repetition_penalty=1.05`. Saved `generation_config.json` has `top_p: 0.8`; demos pass the README values. Card `max_tokens` is 4096; demos use 128.
- **RoPE config.** Current transformers may warn that `rope_parameters` keys (`alpha`, `beta_fast`, `beta_slow`, `mscale`, `mscale_all_dim`) are unrecognized for `rope_type='dynamic'`. Short T4 demos do not use a long window.
- **Float16 on a T4.** Saved dtype is bfloat16. T4 has no native bfloat16. No 4-bit. No FlashAttention-2.
- **Context.** Card `max_context` is 8192. `config.json` also records `max_position_embeddings: 262144`; a T4 KV cache cannot hold a long window on top of 3.6 GB of weights. Keep prompts short.
- **Siblings.** 7B, 30B-A3B, FP8, and GGUF are not loaded here.
- **Published scores are not rerun.** TransRubric / WildMTBench / IFMTBench / API comparisons on the card are the authors' numbers.
- **Apache-2.0.** Public weights; cancel Colab `HF_TOKEN` prompts. This notebook sets `HF_HUB_DISABLE_IMPLICIT_TOKEN=1`.

The next cell drops the model so you can load something else on the same T4. It prints `torch.cuda.max_memory_allocated` for this session.


In [ ]:
print_vram("before cleanup")
print(f"torch.cuda.max_memory_allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
try:
    del model
except NameError:
    pass
try:
    del tokenizer
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print_vram("after cleanup")
    print(f"peak VRAM this session: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Hy-MT3-1.8B released. Re-run the load cell if you want another prompt.")
